<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week8_transformers_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before loading any transformer: what is the difference between a transformer and a traditional ML classifier? Write three differences in plain English — no equations. Check your understanding after the first cell.

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 8 — Transformers: What They Are (STARTER)
### The NLP Internship | LinguaAI Labs

This week: self-attention, fine-tuning DistilBERT on CLINC150, cross-entropy, context windows.

CLINC queries avg 8 words — transformer context window overhead is minimal here vs long documents.

In [ ]:
import sys, subprocess, os, torch
for pkg in ["datasets","transformers"]:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification, Trainer, TrainingArguments)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42); os.makedirs("outputs", exist_ok=True); os.makedirs("models", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device=="cpu": print("CPU ~25-40 min. Colab T4 GPU ~5 min: Runtime→Change runtime type→T4 GPU")

DOMAIN_MAP = {
    "bill_balance":"billing","bill_pay":"billing","transfer":"billing","pay_bill":"billing",
    "account_blocked":"billing","min_payment":"billing",
    "freeze_account":"account","lost_stolen_card":"account","report_fraud":"account",
    "card_declined":"account","cancel_card":"account","credit_score":"account",
    "travel_alert":"travel","flight_status":"travel","book_hotel":"travel","book_flight":"travel",
    "reminder":"utility","calendar":"utility","schedule_meeting":"utility","make_call":"utility",
}
clinc = load_dataset("clinc_oos", "plus")
label_names = clinc["train"].features["intent"].names
def collapse(i): return DOMAIN_MAP.get(label_names[i], "other")
df_train = clinc["train"].to_pandas().rename(columns={"text":"text_clean"})
df_test  = clinc["test"].to_pandas().rename(columns={"text":"text_clean"})
df_train["category"] = df_train["intent"].apply(collapse)
df_test["category"]  = df_test["intent"].apply(collapse)
X_train=df_train["text_clean"].values; y_train_str=df_train["category"].values
X_test =df_test["text_clean"].values;  y_test_str =df_test["category"].values
label2id={l:i for i,l in enumerate(sorted(set(y_train_str)))}
id2label={v:k for k,v in label2id.items()}
y_train=np.array([label2id[l] for l in y_train_str])
y_test =np.array([label2id[l] for l in y_test_str])
print(f"CLINC150 {len(X_train):,} train | {len(label2id)} domains ✅")

## Monday–Tuesday — Self-Attention

> 🧠 **Reflect:** Why can't averaging Word2Vec vectors distinguish 'I want to book a flight' from 'My flight was cancelled — I want a refund'?

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer_vis = AutoTokenizer.from_pretrained(model_name)
model_vis = AutoModel.from_pretrained(model_name, output_attentions=True)
model_vis.eval()
query = "I need to freeze my credit card right away it was stolen"
inputs = tokenizer_vis(query, return_tensors="pt")
with torch.no_grad(): outputs = model_vis(**inputs)
attention = outputs.attentions[-1][0][0].numpy()
tokens = tokenizer_vis.convert_ids_to_tokens(inputs["input_ids"][0])
fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(attention, cmap="Blues")
ax.set_xticks(range(len(tokens))); ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(tokens, fontsize=9)
ax.set_title("Attention — 'freeze my credit card' (account intent)", fontweight="bold", loc="left")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig("outputs/attention_clinc.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
freeze_idx = list(tokens).index("freeze") if "freeze" in tokens else 1
print("\nTop 5 tokens 'freeze' attends to:")
for idx in attention[freeze_idx].argsort()[::-1][:5]:
    print(f"  {tokens[idx]:<15} {attention[freeze_idx][idx]:.3f}")

## Wednesday — Context Windows

> ⏸️ **Predict:** CLINC queries avg 8 tokens. With max_length=128 — what truncation rate do you expect?

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
lengths = [len(bert_tok.tokenize(t)) for t in X_train[:500]]
print(f"Mean: {sum(lengths)/len(lengths):.1f} | Max: {max(lengths)}")
print(f"> 64 tokens:  {sum(1 for l in lengths if l>64)/len(lengths):.1%}")
print(f"> 128 tokens: {sum(1 for l in lengths if l>128)/len(lengths):.1%}")
print("\nFor short-text intent, max_length=64 is often sufficient — try it in the challenge task.")

## Thursday — Fine-Tune DistilBERT on CLINC150

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
def tokenize_fn(ex): return tokenizer(ex["text"],padding="max_length",truncation=True,max_length=128)
train_ds = Dataset.from_dict({"text":list(X_train),"label":list(y_train)}).map(tokenize_fn,batched=True)
test_ds  = Dataset.from_dict({"text":list(X_test), "label":list(y_test)}).map(tokenize_fn,batched=True)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id)
def compute_metrics(ep):
    preds = ep.predictions.argmax(-1)
    return {"f1": f1_score(ep.label_ids,preds,average="weighted",zero_division=0)}
args = TrainingArguments(output_dir="models/distilbert_clinc",num_train_epochs=3,
    per_device_train_batch_size=32,per_device_eval_batch_size=64,
    warmup_steps=100,weight_decay=0.01,evaluation_strategy="epoch",
    save_strategy="epoch",load_best_model_at_end=True,metric_for_best_model="f1",seed=42)
trainer = Trainer(model=bert_model,args=args,train_dataset=train_ds,
                  eval_dataset=test_ds,compute_metrics=compute_metrics)
print("Fine-tuning DistilBERT on CLINC150 (3 epochs)...")
trainer.train(); print("Training complete ✅")

## Cross-Entropy Loss

For 6 classes: random guessing loss ≈ 1.79. A well-trained model should reach below 0.50 by epoch 3.

In [ ]:
history = trainer.state.log_history
train_loss=[(h["step"],h.get("loss",h.get("train_loss")))
            for h in history if ("loss" in h or "train_loss" in h) and "eval_loss" not in h]
eval_loss =[(h["step"],h["eval_loss"]) for h in history if "eval_loss" in h]
eval_f1   =[(h["step"],h["eval_f1"])   for h in history if "eval_f1" in h]

fig,axes=plt.subplots(1,2,figsize=(12,4))
if train_loss: axes[0].plot(*zip(*train_loss),label="Train",color="#2E75B6")
if eval_loss:  axes[0].plot(*zip(*eval_loss),label="Eval",color="#C0392B",marker="o")
axes[0].axhline(1.79,color="gray",linestyle=":",label="Random (6-class)")
axes[0].set_title("Cross-Entropy Loss",fontweight="bold",loc="left")
axes[0].legend(); axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
if eval_f1:    axes[1].plot(*zip(*eval_f1),color="#2E75B6",marker="o")
axes[1].set_title("Weighted F1",fontweight="bold",loc="left")
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/training_curves_clinc.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

preds_out=trainer.predict(test_ds)
y_pred_bert=preds_out.predictions.argmax(-1)
bert_f1=f1_score(preds_out.label_ids,y_pred_bert,average="weighted",zero_division=0)
print(f"DistilBERT on CLINC150: {bert_f1:.3f}")
print(classification_report([id2label[i] for i in preds_out.label_ids],
                              [id2label[i] for i in y_pred_bert],zero_division=0))

## Fine-Tuning vs Prompting

> **Fine-tuning** (this week): update weights on CLINC150 labels — fast inference, no API cost.
> **Prompting**: 'Classify intent as billing/account/travel/utility/other.' No labels needed — but slower, costs per call.
>
> With 15,100 labelled examples and production latency requirements: fine-tuning is correct.

🏆 **Challenge:** Re-run with `max_length=64`. Does F1 change? Is it faster?

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Fine-tuning on a CPU | BERT fine-tuning on CPU takes hours. Always use a GPU runtime in Colab (Runtime → Change runtime type → GPU). | Check torch.cuda.is_available() before starting any training loop |
| Using the full BERT model for short text | BERT has 110M parameters. For support ticket classification, DistilBERT (66M parameters) or a sentence transformer is faster and often equally accurate. | Start with the smallest model that solves the problem |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Run zero-shot classification using a pre-trained transformer on your ticket categories. What accuracy does it achieve without any fine-tuning?

> 🔬 **Advanced Path**
> Fine-tune DistilBERT on your training set for 3 epochs. Compare its F1 to your Week 5 TF-IDF baseline. Is the improvement worth the compute cost?

## ✅ What You Can Do After This Week

- Visualise BERT attention on real intent queries
- Fine-tune DistilBERT on CLINC150 in 3 epochs
- Interpret cross-entropy training curves (random baseline = 1.79)
- Explain fine-tuning vs prompting tradeoff

---
## ✅ Week 8 Complete — Next: `week9_fine_tuning_STARTER.ipynb`

---
*Add `week8_transformers_STARTER.ipynb` to your portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 8, you can now:

- Explain what attention does in a transformer in one plain-English sentence
- Run a pre-trained transformer for zero-shot text classification
- Compare transformer and traditional ML performance on the same task
- Explain when a simpler model is the better professional choice

📤 **GitHub:** Push week8_transformers_STARTER.ipynb. Commit: "Week 8: Transformers — zero-shot baseline established"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
